# Statistical Parametric Mapping (SPM)

**Author**: Steffen Bollmann, Michèle Masson-Trottier

The University of Queensland

<div style="line-height: 2;">
<a href="https://github.com/stebo85"><img src="https://img.shields.io/badge/-Steffen_Bollmann-181717?logo=github" alt="GitHub"></a> <a href="https://orcid.org/0000-0002-2909-0906"><img src="https://img.shields.io/badge/ORCID-0000--0002--2909--0906-green?logo=orcid" alt="ORCID"></a><br>
<a href="https://github.com/micmas"><img src="https://img.shields.io/badge/-Michèle_Masson--Trottier-181717?logo=github" alt="GitHub"></a> <a href="https://orcid.org/0000-0002-0642-5662"><img src="https://img.shields.io/badge/ORCID-0000--0002--0642--5662-green?logo=orcid" alt="ORCID"></a>
</div>

**Date**: 30/03/2026

**License:** 
<div style="margin-top: 10px;">
    <a href="https://creativecommons.org/licenses/by/4.0/" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> CC-BY-4.0 License
    </a>
</div>

## Purpose

Tutorial for running a functional MRI analysis in SPM — from data download through preprocessing to first-level GLM analysis. Based on [Andy's Brain Book SPM tutorial](https://andysbrainbook.readthedocs.io/en/latest/SPM/SPM_Overview.html), adjusted for the Neurodesk platform.

:::{admonition} Learning Objectives
:class: tip

After completing this tutorial, you will be able to:

- Download and prepare an fMRI dataset using DataLad
- Perform standard SPM preprocessing (realignment, slice timing, coregistration, segmentation, normalisation, smoothing)
- Set up and estimate a first-level GLM
- Interpret activation maps from a contrast of interest

:::

## Citation and Resources

**SPM**
: Friston, K.J., et al. (1994). Statistical parametric maps in functional imaging: A general linear approach. *Human Brain Mapping*, 2(4), 189–210. https://doi.org/10.1002/hbm.460020402

**Dataset (ds000102)**
: Kelly, A.M.C., et al. (2008). Competition between functional brain networks mediates behavioral variability. *NeuroImage*, 39, 527–537. https://doi.org/10.1016/j.neuroimage.2007.08.008

**Educational Resources**
: [Andy's Brain Book — SPM Overview](https://andysbrainbook.readthedocs.io/en/latest/SPM/SPM_Overview.html), [SPM documentation](https://www.fil.ion.ucl.ac.uk/spm/docs/)

## Prerequisites

:::{admonition} Requirements
:class: warning

- Running Neurodesk
- Familiarity with basic fMRI concepts
- Access to the internet to download example data (ds000102)

:::

## Section 1: Download Data

We will use the open dataset ds000102 (Flanker task). Open a terminal and use DataLad to install the dataset:

```bash
cd ~/neurodesktop-storage/
datalad install https://github.com/OpenNeuroDatasets/ds000102.git
cd ds000102
datalad get sub-08/
gunzip sub-08/anat/sub-08_T1w.nii.gz -f
gunzip sub-08/func/sub-08_task-flanker_run-1_bold.nii.gz -f
gunzip sub-08/func/sub-08_task-flanker_run-2_bold.nii.gz -f
chmod a+rw sub-08/ -R
```

:::{note}
SPM doesn't support compressed NIfTI files, so we decompress them. For details on the Flanker task, see [Andy's Brain Book](https://andysbrainbook.readthedocs.io/en/latest/SPM/SPM_Short_Course/SPM_02_Flanker.html).
:::

## Section 2: Start SPM and Visualise Data

**Step 1**: Start spm12GUI from the Neurodesk Application Menu.

**Step 2**: Select 'fMRI' from the modality selection screen.

**Step 3**: Change the SPM working directory to the sub-08 folder using `Utils` → `CD`.

**Step 4**: Display the anatomical T1 scan using `Display` and navigate to select the T1 image.

**Step 5**: Check alignment before preprocessing using `CheckReg`:
- Open the anatomical scan
- Open the functional scan (Frames 1:146)
- Note: They will not align yet (expected — we haven't preprocessed)

![Displaying data in SPM](/static/tutorials/functional_imaging/spm/02_spm_fmri.png)
*The SPM12 interface with fMRI mode selected.*

## Section 3: Preprocessing — Realignment

1. Select `Realign (Est & Reslice)` from the SPM Menu
2. Select the functional run (select frames 1:146)
3. Leave all other settings as defaults
4. Press the green "Play" button
5. Examine the realignment parameters plot showing head motion across volumes

![Realignment output showing head motion parameters](/static/tutorials/functional_imaging/spm/07_realign_output.png)
*Realignment parameters showing head motion across volumes.*

## Section 4: Preprocessing — Slice Timing Correction

1. Click on `Slice timing` in the SPM menu
2. Select the realigned images (filter for `rsub`, frames 1:146)
3. Enter the parameters:
   - Number of Slices = 40
   - TR = 2
   - TA = 1.95
   - Slice order = [1:2:40 2:2:40]
   - Reference Slice = 1
4. Press the green "Play" button

![Slice timing parameters](/static/tutorials/functional_imaging/spm/08_slicetiming.png)
*Configuring slice timing parameters in the SPM Batch Editor.*

## Section 5: Preprocessing — Coregistration

1. Click on `Coregister (Estimate & Reslice)` in the SPM menu
2. Use the Mean image as the reference
3. Use the T1 scan as the source image
4. Press the green "Play" button
5. Verify alignment using `CheckReg` with a Contour overlay (Right Click → Contour → Display onto → all)

![CheckReg with contour overlay after coregistration](/static/tutorials/functional_imaging/spm/10_checkreg_contour.png)
*CheckReg with contour overlay confirming successful coregistration.*

## Section 6: Preprocessing — Segmentation

1. Click on `Segmentation` in the SPM menu
2. Set the following parameters:
   - Volumes = the coregistered anatomical scan (rsub-08_T1w.nii)
   - Save Bias Corrected = checked
   - Deformation Fields = Forward
3. Press the green "Play" button

## Section 7: Preprocessing — Normalisation and Smoothing

**Normalisation:**
1. Select `Normalize (Write)` from the SPM menu
2. For Deformation Field, select the y_rsub-08 file created in the segmentation step
3. For Images to Write, select the arsub-08 functional images (filter `^ar`, frames 1:146)
4. Press the green "Play" button
5. Verify normalisation using `CheckReg` against the MNI template at `/opt/spm12/spm12_mcr/spm12/spm12/canonical/avg305T1.nii`

![CheckReg verifying normalisation against MNI template](/static/tutorials/functional_imaging/spm/12_normalise_check.png)
*Verifying normalisation by overlaying functional image on the MNI template.*

**Smoothing:**
1. Click the `Smooth` button in the SPM menu
2. Select the warped functional scans (starting with `w`)
3. Press the green "Play" button

![CheckReg after smoothing](/static/tutorials/functional_imaging/spm/13_smooth_check.png)
*CheckReg after smoothing.*

## Section 8: First-Level GLM Setup

1. Click on `Specify 1st-level` in the SPM menu
2. Set the following options:
   - **Directory**: Select the sub-08 top-level directory
   - **Units for design**: Seconds
   - **Interscan interval (TR)**: 2
3. Click twice on `New Subject/Session` to create two sessions
4. For each session, select the smoothed, warped data (frames 1:146)
5. Create two conditions per session:

**Run 1:**
- Condition 1: Inc (Incongruent)
  - Onsets: 0 10 20 52 88 130 144 174 236 248 260 274
  - Durations: 2
- Condition 2: Con (Congruent)
  - Onsets: 32 42 64 76 102 116 154 164 184 196 208 222
  - Durations: 2

**Run 2:**
- Condition 1: Inc (Incongruent)
  - Onsets: 0 10 52 64 88 150 164 174 184 196 232 260
  - Durations: 2
- Condition 2: Con (Congruent)
  - Onsets: 20 30 40 76 102 116 130 140 208 220 246 274
  - Durations: 2

6. Press the green "Play" button

![First-level GLM design matrix](/static/tutorials/functional_imaging/spm/14_design_matrix.png)
*The first-level GLM design matrix showing the two conditions across two runs.*

## Section 9: Estimate the Model

1. Click on `Estimate` in the SPM menu
2. Select the SPM.mat file created in the previous step
3. Press the green "Play" button
4. (Optional) Review the design matrix by clicking `Review` and selecting the SPM.mat file

## Section 10: Inference

1. Click on `Results` in the SPM menu
2. Select the SPM.mat file
3. Define a new contrast:
   - **Name**: Incongruent-Congruent
   - **Contrast weights vector**: 0.5 -0.5 0.5 -0.5
4. View results with the following options:
   - **Masking**: None
   - **p value adjustment**: None (uncorrected)
   - **Uncorrected p-value**: 0.01
   - **Extent threshold**: 10 voxels
5. Press "Apply" to view the activation map

![Activation map for Incongruent-Congruent contrast](/static/tutorials/functional_imaging/spm/15_results.png)
*Activation map for the Incongruent > Congruent contrast.*

## Summary

You have successfully:

- Downloaded the ds000102 Flanker task dataset using DataLad
- Performed SPM preprocessing:
  - Realignment (motion correction)
  - Slice timing correction
  - Coregistration (anatomical to functional)
  - Segmentation (tissue classification)
  - Normalisation (to MNI template)
  - Smoothing (8mm FWHM)
- Estimated a first-level GLM with two conditions (incongruent and congruent)
- Generated activation maps for the Incongruent > Congruent contrast

You can now interpret these results and proceed to second-level analyses across participants.

## See Also

- [fMRIPrep Tutorial](./fmriprep.ipynb) — Automated preprocessing alternative to manual SPM steps
- [PhysIO Tutorial](./physio.ipynb) — Physiological noise correction for improved GLM models
- [MRIQC Tutorial](./mriqc.ipynb) — Quality control assessment before preprocessing
- [SPM Official Documentation](https://www.fil.ion.ucl.ac.uk/spm/docs/) — Comprehensive reference for all SPM functions
- [Andy's Brain Book](https://andysbrainbook.readthedocs.io/en/latest/SPM/SPM_Overview.html) — Extended SPM tutorials